# Advanced NumPy — Task 14 Solutions

This notebook holds the worked answers to the nine Task 14 problems, each paired with a note on the technique it showcases. Use it after attempting the questions in the task notebook — or whenever you want to compare a clean solution with yours. The problems here are chosen to exercise exactly the advanced toolkit from Session 14: absolute-difference minimisation, boolean masks, fancy indexing, missing-data handling, broadcast-constructed matrices, vectorised plotting, and the implementation of the broadcasting rules.

As you read each solution, watch for the one-line 'core idea':

- Q1 is `abs(arr - a).argmin()` — distance as a vector, reduction to the best index.
- Q2 is a combined boolean mask assigned to zero.
- Q3 shows fancy indexing in two modes: writing (`*=2`) and gathering (repeated indices).
- Q4 and Q5 handle the missing data choreography with `np.isnan` and `np.nan_to_num`.
- Q6 builds an entire matrix from one broadcast of a column minus a row.
- Q7 and Q8 turn equations into plots purely with `np.linspace` and `plt.plot`.
- Q9 turns the two broadcasting rules into executable logic.

A couple of these reference solutions are intentionally left incomplete or minimal — notably Q9, where the checker's final return is left for you — mirroring real interviews where a candidate sketches the approach and the interviewer fills in details. Treat the notes below combined with your own attempts as the full answer. Read in order, because each problem reuses the vocabulary of the ones before it.

Two habits will make the most of this notebook. First, study shape at every step: ask what shape each intermediate array has and why, since broadcasting questions live and die by shapes. Second, after reading a solution, rewrite the core line from memory on a fresh cell — the transition from 'I recognise it' to 'I can produce it' happens exactly at that moment. Learning the advanced toolkit is not memorising functions but building the reflex for when each one applies. Let's walk through them.

In [ ]:
import numpy as np

### `Q-1:` Find the nearest element in the array to a given integer.
```
Ex:- 
a=23 and array - [10 17 24 31 38 45 52 59].
Nearest element is 24
```
Hint: Read about this function `argmin()`

## 1.1 Solution notes — Q1: Nearest element

`arr = np.array([10, 17, 24, 31, 38, 45, 52, 59])` and `a = 39`. The vector `abs(arr - a)` holds every distance at once: `[29, 22, 15, 8, 1, 6, 13, 20]`. `np.argmin` returns the index of the smallest distance — position 4 — and `arr[...]` retrieves 38. The pattern — broadcast a scalar, take absolute values, reduce with argmin, then index back — is the stock recipe for nearest-neighbour and closest-point problems.

In [ ]:
# code here
arr = np.array([10, 17, 24, 31, 38, 45, 52, 59])
a=39
arr[abs(arr - a).argmin()]

38

### `Q-2:` Replace multiples of 3 or 5 as 0 in the given array.
```
arr=[1 2 3 4 5 6 7 9]

result-> [1 2 0 4 0 0 7 0]
```

## 2.1 Solution notes — Q2: Multiples of 3 or 5 to zero

Two boolean masks, one elementwise OR: `(arr % 3 == 0) | (arr % 5 == 0)` marks 3, 5, 6, and 9 — 1 and 7 stay False. The parentheses are essential because comparison binds tighter than the bitwise-or used for arrays, and `|` (not Python's `or`) is the vectorised aggregate. `arr[mask] = 0` then writes zeros into exactly those slots, giving `[1 2 0 4 0 0 7 0]`. This is the canonical conditional-replacement idiom.

In [ ]:
# code here
arr=np.array([1, 2, 3, 4, 5, 6, 7, 9])
arr[(arr%3==0) | (arr%5==0)] = 0
arr

array([1, 2, 0, 4, 0, 0, 7, 0])

### `Q-3:` Use Fancy Indexing.
1. Double the array elements at  given indexes
```
arr = np.arrange(10)
indexes = [0,3,4,9]
```
Result -> `[ 0  1  2  6  8  5  6  7  8 18]`

2. Using a given array make a different array as in below example
```
array = [1,2,3]
result array -> [1 1 1 2 2 2 3 3 3]
```
* Internal-repetion should be as length of the array.

Hint: 
```
if a is an array
a = [2,4]
a[[1,1,0,1]] will result in-> [4 4 2 4]
```

## 3.1 Solution notes — Q3a: Fancy indexing, writing mode

Part 1 doubles the elements at indexes 0, 3, 4, 9 with `arr[indexes] *= 2`. Fancy indexing here is used for assignment: the integer list selects the target positions and the in-place multiplication writes them back. The untouched positions (1, 2, 5, 6, 7, 8) keep their values, producing `[0 1 2 6 8 5 6 7 8 18]`. The `indexes` list reads as 'these indices, exactly these'. Contrast this with the reading-mode of part 2.

In [ ]:
# code here
#1
arr = np.arange(10)
indexes = [0,3,4,9]
arr[indexes] *=2
arr

array([ 0,  1,  2,  6,  8,  5,  6,  7,  8, 18])

## 3.2 Solution notes — Q3b: Fancy indexing, gathering mode

Part 2 builds the index list `[0, 0, 0, 1, 1, 1, 2, 2, 2]` with a loop appending each position `len(array)` times, then evaluates `array[indexes]`. Fancy indexing gathers values in the order the indices appear, and repeats are honoured — so `array = [1 2 3]` yields `[1 1 1 2 2 2 3 3 3]`. The loop only builds integers; the actual reordering is a pure NumPy gather. Reading 'index list determines order and multiplicity' is the concept.

In [ ]:
array = np.array([1,2,3])
indexes = []
for index in range(len(array)):
    indexes.extend([index] * len(array))
array[indexes]

array([1, 1, 1, 2, 2, 2, 3, 3, 3])

### `Q-4:` Your are given an array which is havig some nan value. You job is to fill those nan values with most common element in the array.
```
arr=np.array([[1,2,np.nan],[4,2,6],[np.nan,np.nan,5]])

```

## 4.1 Solution notes — Q4: NaN filled with the mode

The dictionary loop counts every value over `arr.flatten()`, producing counts where 2 appears four times, 1 twice, and 4, 5, 6 once each. Sorting the items by count (`sorted(d.items(), key=lambda x: x[1])[-1][0]`) picks the most frequent key — 2. Then `arr[np.isnan(arr)] = 2` sets all four NaN cells to 2. The result is `[[1 2 2] [4 2 6] [2 2 5]]`. Two lessons: a frequency count done by hand is just `np.unique(return_counts=True)`, and boolean masks assign into any selection shape.

In [ ]:
# code here
arr=np.array([[1,2,np.nan],[4,2,6],[np.nan,np.nan,5]])
d = {}
for i in arr.flatten():
    if i in d:
        d[i] += 1
    else:
        d[i] = 1

arr[np.isnan(arr)] = sorted(d.items(), key = lambda x: x[1])[-1][0]
arr

array([[1., 2., 2.],
       [4., 2., 6.],
       [2., 2., 5.]])

### `Q-5:` Write a NumPy program 

- to find the missing data in a given array. Return a boolean matrix.
- also try to fill those missing values with 0. For that, you can use `np.nan_to_num(a)`

```python
import numpy as np

np.array([[3, 2, np.nan, 1],
          [10, 12, 10, 9],
          [5, np.nan, 1, np.nan]])
```

## 5.1 Solution notes — Q5a: Missing-data mask

`np.isnan(arr)` returns a boolean matrix of the same shape with True exactly where NaN sits — for this array, at `[0, 2]`, `[2, 1]`, and `[2, 3]`. This is the answer to the first part: a boolean matrix that flags every missing cell. No cleanup happens yet — detection is the whole point of part 1.

In [ ]:
# code here
#1
arr = np.array([[3, 2, np.nan, 1],
          [10, 12, 10, 9],
          [5, np.nan, 1, np.nan]])
np.isnan(arr)

array([[False, False,  True, False],
       [False, False, False, False],
       [False,  True, False,  True]])

## 5.2 Solution notes — Q5b: Fill with zeros

`np.nan_to_num(arr)` replaces every NaN with 0 in the returned copy, turning the array into a clean float matrix where the missing spots read 0. The result reflows the row with `[3. 2. 0. 1.]` and the last row's two NaNs become `[5. 0. 1. 0.]`. `nan_to_num` is the one-call cleanup when you want a neutral numeric stand-in rather than an estimate like the mode or mean.

In [ ]:
#2
np.nan_to_num(arr)

array([[ 3.,  2.,  0.,  1.],
       [10., 12., 10.,  9.],
       [ 5.,  0.,  1.,  0.]])

### `Q-6:` Given two arrays, X and Y, construct the Cauchy matrix C.
`Cij =1/(xi - yj)`

http://en.wikipedia.org/wiki/Cauchy_matrix
```
x = numpy.array([1,2,3,4]).reshape((-1, 1)
y = numpy.array([5,6,7])
```

## 6.1 Solution notes — Q6: Cauchy matrix

`x = np.array([1,2,3,4]).reshape((-1, 1))` becomes a (4,1) column, `y = [5,6,7]` stays (3,). In `1/(x-y)`, broadcasting expands the column against the row into a full (4,3) grid of differences, then the reciprocal fills each cell — `C[i][j] = 1/(xi - yj)`. No loops: the entire matrix is one broadcasted expression. The sampled output shows the four-by-three grid with values like -0.25 at the top-left (`1/(1-5)`). Matrices built this way never allocate intermediate grids by hand — a hallmark of broadcast fluency.

In [ ]:
# code here
x = np.array([1,2,3,4]).reshape((-1, 1))
y = np.array([5,6,7])

1/(x-y)

array([[-0.25      , -0.2       , -0.16666667],
       [-0.33333333, -0.25      , -0.2       ],
       [-0.5       , -0.33333333, -0.25      ],
       [-1.        , -0.5       , -0.33333333]])

### `Q-7:` Plot this below equation.

$$\large y = \frac{e^x - e^{-x}}{e^x + e^{-x}}$$

**Note: This equation is called tanh activation function. In deep learning, many times this function is used. If you find some difference between the sigmoid function and this tanh function, note that to your notebook.**

## 7.1 Solution notes — Q7: Plotting tanh

`x = np.linspace(-30, 30, 1000)` provides a dense smooth domain, and the tanh formula `(np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))` evaluates vectorised — three exponential and two arithmetic passes over the whole array. `plt.plot(x, y)` draws the classical S-curve that asymptotes to -1 and +1 and passes symmetrically through the origin. Compare with sigmoid in your notebook: tanh is centred at zero and ranges (-1, 1), whereas sigmoid ranges (0, 1); that centring is why hidden layers often prefer tanh. With 1000 points the curve is perfectly smooth.

In [ ]:
# code here
import matplotlib.pyplot as plt

x = np.linspace(-30, 30, 1000)
y = (np.exp(x) - np.exp(-x)) / (np.exp(x) + np.exp(-x))

plt.plot(x,y)

<Figure size 640x480 with 1 Axes>

### `Q-8:` Plot the below equation.

$$\large y = \sqrt{36 - (x - 4)^2} + 2$$

The range of x should be between -2 to 10. $x ∈ [-2, 10]$



## 8.1 Solution notes — Q8: Semicircle plot

`x = np.linspace(-2, 10, 100)` covers the permitted domain, and `y = np.sqrt(36 - (x - 4)**2) + 2` is fully vectorised. The equation rearranges to `(x-4)^2 + (y-2)^2 = 36`, a circle of radius 6 centred at (4, 2); taking the positive root plots only the upper semicircle. The plot output confirms a dome that reaches y = 8 at x = 4 (2 + radius). Note the formula stays over the domain where the radicand is non-negative — endpoints -2 and 10 give height 2.

In [ ]:
# code here
x = np.linspace(-2, 10, 100)
y = np.sqrt(36 - (x-4)**2) + 2

plt.plot(x,y)

<Figure size 640x480 with 1 Axes>

### `Q-9:` Write a program implement Broadcasting Rule to check if two array can be added or not.
Given tuples of shapes.
```
shape of a- (3,2,2)
shape of b- (2,2)

check_broadcast(a, b) -> return Boolean (True if can broadcasted, False other wise.)
```

## 9.1 Solution notes — Q9: Broadcasting checker

The solution sketches the algorithm in comments: align both shape tuples from the right, then walk pairs — reversed tuples are zipped so trailing dimensions are compared first. If any pair of dimensions is unequal and neither is 1, broadcasting is impossible; the `if` on mismatched non-1 dimensions should return False. The final `return` (True as default) is left blank for you to complete — the intended `return True` concludes the check. For `(3,2,2)` vs `(2,2)`, the trailing `(2,2)` matches and the leading 3 pairs with a phantom 1, so broadcasting succeeds. Finishing this function is the true final test.

## Summary

This solutions notebook completed all nine Task 14 problems. `abs(arr - a).argmin()` located the nearest value; `(arr % 3 == 0) | (arr % 5 == 0)` composed a combined mask that zeroed select entries; and fancy indexing worked in both writing and gathering modes, doubling chosen positions and materialising repeated patterns. Missing data was detected with `np.isnan`, neutralised with `np.nan_to_num`, and imputed with the most frequent value after a frequency count. One broadcasted `1/(x-y)` built the Cauchy matrix, and two vectorised equations became plots — tanh and an upper semicircle. The final exercise translated the broadcasting rules into a checker, leaving one return for you to complete. Re-attempt the unfinished parts and the earlier tasks from memory — every one of these patterns belongs in your daily toolkit.

In [ ]:
# code here

# 4 3 3 5
#     2 5

# Align both shapes
# Loop for both shapes from behind
#     If two numbers are not equal and none of the numbers are 1, return false
# return true

def check_broadcast(a,b):
    a = a[::-1]
    b = b[::-1]
    for ai, bi in zip(a,b):
        if ai != bi and a1 != 1 and b1 != 1:
            return False
    return